# PREPROCESSING

## Data Loading

In [ ]:
import pandas as pd
import numpy as np
import glob

def load_dataset(folder):
    all_files = glob.glob(f"{folder}/**/*.csv", recursive=True)
    samples = []

    print(f"Found {len(all_files)} files in '{folder}'")

    for i, filepath in enumerate(all_files, start=1):

        if i % 500 == 0 or i == 1:
            print(f"[{folder}] Processed {i}/{len(all_files)} files")

        df = pd.read_csv(filepath)

        file_id = df['file_id'].iloc[0]
        label = df['label'].iloc[0] if 'label' in df.columns else None

        features = {}

        for col in ['mean_x', 'mean_y', 'mean_z', 'std_x', 'std_y', 'std_z']:
            features[f'{col}_mean'] = df[col].mean()
            features[f'{col}_std'] = df[col].std()
            features[f'{col}_min'] = df[col].min()
            features[f'{col}_max'] = df[col].max()
            features[f'{col}_median'] = df[col].median()

            # extra features
            features[f'{col}_range'] = df[col].max() - df[col].min()
            features[f'{col}_q25'] = df[col].quantile(0.25)
            features[f'{col}_q75'] = df[col].quantile(0.75)

        mag = np.sqrt(
            df['mean_x']**2 +
            df['mean_y']**2 +
            df['mean_z']**2
        )

        features['mag_mean'] = mag.mean()
        features['mag_std'] = mag.std()
        features['mag_min'] = mag.min()
        features['mag_max'] = mag.max()

        features['file_id'] = file_id
        features['label'] = label

        samples.append(features)

    print(f"Finished loading '{folder}'")

    return pd.DataFrame(samples)


train_df = load_dataset("train")
test_df = load_dataset("test")

print(f"Train: {train_df.shape}")
print(f"Test:  {test_df.shape}")

Train: (11020, 32), Test: (6849, 32)


## Preparation of X,Y and normalization

In [ ]:
feature_cols = [c for c in train_df.columns
                if c not in ['file_id', 'label']]

X_train = train_df[feature_cols].values
y_train = train_df['label'].values
X_test = test_df[feature_cols].values

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")

X_train shape: (11020, 30)
X_test shape:  (6849, 30)


## XGBoost

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import cross_val_score

xgb = XGBClassifier(
    n_estimators=200,
    learning_rate=0.1,
    random_state=42,
    eval_metric='mlogloss'
)

scores = cross_val_score(xgb, X_train, y_train,
                         cv=5, scoring='f1_macro')
print(f"XGBoost F1-macro: {scores.mean():.4f}")

XGBoost F1-macro: 0.6661
